# Task 2 – History and calendar features (Maram)
**Input:** `food_clean.csv` (from `Task2_Cleaning.ipynb`, run that first)  
**Output:** `maram_features.csv`, the history and calendar columns, joined with Joud's columns at the end of the week.

In [ ]:
import pandas as pd

We only need pandas here.

In [ ]:
df = pd.read_csv('food_clean.csv')
df['Inspection Date'] = pd.to_datetime(df['Inspection Date'])
print(df.shape)

We read the clean file. The date becomes text again when saved to CSV, so we change it back to a real date.

In [ ]:
# M1: sort each business's inspections from oldest to newest
df = df.sort_values(by=['License #', 'Inspection Date'])
df[['License #', 'Inspection Date', 'Results']].head(10)

**M1.** We sort by License # and then by date, so the inspections of one business are next to each other, oldest first. The next steps need this order.

In [ ]:
# M2: did this business fail its previous inspection?
df['Prev_Fail'] = df.groupby('License #')['Fail'].shift(1)
df[['License #', 'Inspection Date', 'Fail', 'Prev_Fail']].head(10)

**M2.** `shift(1)` is new for us: it moves the column down by one row, so each inspection gets the result of the inspection before it. `groupby('License #')` makes sure it only looks inside the same business. The first inspection of each business has nothing before it, so it gets an empty value (NaN).

In [ ]:
df['First_Inspection'] = df['Prev_Fail'].isnull().apply(lambda x: 1 if x else 0)
df['Prev_Fail'] = df['Prev_Fail'].fillna(0)
df['First_Inspection'].value_counts()

We save "is this the first inspection we have for this business?" in its own column (1 = yes), then fill the empty Prev_Fail with 0. This way the model still knows which rows had no history. (Write here how many first inspections there are.)

In [ ]:
# M3: days since the last inspection of the same business
df['Days_Since_Last'] = df.groupby('License #')['Inspection Date'].diff().dt.days
df['Days_Since_Last'].describe()

**M3.** `diff()` and `.dt.days` are new for us: `diff()` subtracts the previous date from this date (inside the same business), and `.dt.days` turns the result into a number of days. First inspections have no previous date, so they are empty. (Write here the median number of days.)

In [ ]:
median_days = df['Days_Since_Last'].median()
df['Days_Since_Last'] = df['Days_Since_Last'].fillna(median_days)
print(median_days)

We fill the empty values (first inspections) with the median, because the median is not affected by very long gaps. The First_Inspection column still tells the model these rows were filled.

In [ ]:
# M4: month of the inspection
df['Month'] = df['Inspection Date'].dt.month
df['Month'].value_counts().sort_index()

**M4.** The month (1–12). Hot months may have more temperature problems; the Chicago model also used temperature.

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['Season'] = df['Month'].apply(get_season)
df['Season'].value_counts()

We group the months into seasons with a def + apply, like we did in the labs.

In [ ]:
# M5: keep the 8 most common facility types, the rest becomes 'Other'
top8 = df['Facility Type'].value_counts().head(8).index
top8

**M5.** There are many facility types and most have very few rows. We keep the 8 most common ones (written above).

In [ ]:
def group_facility(t):
    if t in top8:
        return t
    else:
        return 'Other'

df['Facility_Group'] = df['Facility Type'].apply(group_facility)
df['Facility_Group'].value_counts()

Every other type becomes "Other", the same idea as grouping small categories in the Kindle assignment. Later, for the model, we will turn this column into numbers with `get_dummies`.

In [ ]:
# M6: risk as a number
def risk_number(r):
    if r == 'Risk 1 (High)':
        return 1
    elif r == 'Risk 2 (Medium)':
        return 2
    elif r == 'Risk 3 (Low)':
        return 3
    else:
        return 0

df['Risk_Num'] = df['Risk'].apply(risk_number)
df['Risk_Num'].value_counts()

**M6.** The model needs numbers, so Risk 1 (High) = 1, Risk 2 = 2, Risk 3 (Low) = 3, and Unknown = 0.

In [ ]:
# M7: placebo period: a fake change date (1 Jan 2017) inside the old-code years
def placebo_period(date):
    if date < pd.Timestamp('2017-01-01'):
        return 'Before 2017'
    else:
        return 'After 2017'

df['Placebo_Period'] = df['Inspection Date'].apply(placebo_period)
df['Placebo_Period'].value_counts()

**M7.** For H3 we compare the real change (July 2018) with a fake one (January 2017), when nothing changed. If the model drops much more at 2018 than at 2017, the drop is caused by the new code. (Period with the grace period is already made in the cleaning notebook.)

In [ ]:
# Check: Prev_Fail against this result
pd.crosstab(df['Prev_Fail'], df['Fail'], normalize='index').round(3)

Quick check: if businesses that failed last time fail more often this time, Prev_Fail is a useful feature. (Write here the fail rate for Prev_Fail = 0 and = 1.)

In [ ]:
cols = ['Inspection ID', 'Prev_Fail', 'First_Inspection', 'Days_Since_Last',
        'Month', 'Season', 'Facility_Group', 'Risk_Num', 'Placebo_Period']
df[cols].to_csv('maram_features.csv', index=False)
print(df[cols].shape)

We save only the ID and the new columns. At the end of the week this file is joined with Joud's file and the clean file on `Inspection ID`.